# 4.0.3-Embedding of all cohort

**Inés Sentís**

Date of execution

In [ ]:
Sys.Date()

## Libraries

In [ ]:
suppressMessages(suppressWarnings({
library(Seurat)
library(Signac)
library(tidyverse)
library(harmony)
library(here)
library(glue)
}))

## Parameters

In [ ]:
#here::dr_here(show_reason = TRUE)
source(here("utils/bin.R"))
source(here("utils/paths.R"))
source(here("AVGUSTINOVA_02_to_30/utils/style.R"))
source(here("AVGUSTINOVA_02_to_30/utils/variables.R"))
set.seed(0)

In [ ]:
args <- commandArgs(TRUE)
op <- args[2]
op

## Load Data

In [ ]:
combined_obj <- readRDS(here(glue("{int}/{robj_dir}/4.0.2-combined_object_{op}.rds")))

In [ ]:
combined_obj

In [ ]:
rna_comp = 1:50
atac_comp = 2:20

### Joint (WNN)

In [ ]:
combined_obj <- FindMultiModalNeighbors(
 combined_obj,
 reduction.list = list("pca", "lsi.common"),
 dims.list = list(rna_comp, atac_comp)
)

In [ ]:
combined_obj <- RunUMAP(
   combined_obj,
   nn.name = "weighted.nn",
   reduction.name = "wnn.umap",
   reduction.key = "wnnUMAP_")

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 6)
p1 <- DimPlot(
  combined_obj,
   reduction = "wnn.umap",
   group.by = "clusters_donor",
   pt.size = 0.1,
   raster=FALSE
) + ggtitle('Joint UMAP WNN\nby clusters (no integration)') + 
    theme(plot.title = element_text(hjust = 0.5, size = 12, face = "bold"))

p2 <- DimPlot(
  combined_obj,
   reduction = "wnn.umap",
   group.by = "subproj",
   pt.size = 0.1,
   raster=FALSE
) + ggtitle('Joint UMAP WNN\nby subproject (no integration)') + 
    theme(plot.title = element_text(hjust = 0.5, size = 12, face = "bold"))

patchwork::wrap_plots(list(p1,p2), ncol=2)

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 6)
p1 <-DimPlot(
  combined_obj,
   reduction = "wnn.umap",
   group.by = "patient",
   cols= colors_patients, 
   pt.size = 0.1,
   raster=FALSE
) + ggtitle('Joint UMAP WNN\nby donor (no integration)') + 
    theme(plot.title = element_text(hjust = 0.5, size = 12, face = "bold")) 

p2 <- DimPlot(
  combined_obj,
   reduction = "wnn.umap",
   group.by = "cell_groups",
   pt.size = 0.1,
   cols=colors_cellg,
   raster=FALSE
) + ggtitle('Joint UMAP WNN\nby cell groups (no integration)') + 
    theme(plot.title = element_text(hjust = 0.5, size = 12, face = "bold"))

patchwork::wrap_plots(list(p1,p2), ncol=2)

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 10)
FeaturePlot(
  combined_obj,
   reduction = "wnn.umap",
   features = c("PTPRC","CD3D","CD8A","CD14","CD68", "COL4A2"),
   pt.size = 0.1,
   min.cutoff = 'q5',
   max.cutoff = 'q95',
   order=T,
   raster=FALSE,
   ncol=3)

In [ ]:
saveRDS(combined_obj, here(glue("{int}/{robj_dir}/4.0.3-combined_object_unify_{op}.rds")))

## Integration with Harmony

### RNA

In [ ]:
# make a copy of the PCA reduction
combined_obj@reductions$pca_filt <- combined_obj@reductions$pca

# Filter to the desired number of components according to elbow
combined_obj@reductions$pca_filt@cell.embeddings <- combined_obj@reductions$pca_filt@cell.embeddings[, rna_comp]

combined_obj@reductions$pca_filt@feature.loadings <- combined_obj@reductions$pca_filt@feature.loadings[, rna_comp]


combined_obj <- RunHarmony(
  object = combined_obj,
  reduction = "pca_filt",
  dims = rna_comp,
  group.by.vars = c("patient"),
  assay.use = "RNA",
  project.dim = T,
  reduction.save = "harmony_pca"
)

#### PCA regression

In [ ]:
df_info <- read_csv(here("utils/info_samples.csv"),show_col_types = FALSE)
df_clinical <- df_info[df_info$modality == "multiome",]
df_clinical <- df_clinical[,c("patient", "subproject", "quality", "location", "age_simple", 
                              "methylome", "alteration_status", "treatment_status")]

In [ ]:
df_tmp <- combined_obj@meta.data %>%
    rownames_to_column("barcodes") %>%
    left_join(df_clinical, by = "patient") %>%
    column_to_rownames("barcodes")

In [ ]:
#integrated_obj@meta.data <- df_tmp
copy_obj <- combined_obj
copy_obj@meta.data <- df_tmp

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 6)

pcr_plot <- .plotPcr(
   obj = copy_obj, 
   var_list = covariates, 
   color_palette = cov_colors, 
   reduction = "harmony_pca",
   comp = 1:20
 )
pcr_plot

In [ ]:
remove(copy_obj)
remove(df_clinical)

In [ ]:
combined_obj <- RunUMAP(
  combined_obj,
  dims = rna_comp,
  reduction = "harmony_pca",
  reduction.name = "umap.rna.harmony",
  reduction.key = "RNAhar_",
  assay = "RNA")

### ATAC

In [ ]:
# make a copy of the reduction
combined_obj@reductions$lsi.common_filt <- combined_obj@reductions$lsi.common

# Filter to the desired number of components according to elbow
combined_obj@reductions$lsi.common_filt@cell.embeddings <- combined_obj@reductions$lsi.common_filt@cell.embeddings[, atac_comp]

combined_obj@reductions$lsi.common_filt@feature.loadings <- combined_obj@reductions$lsi.common_filt@feature.loadings[, atac_comp]

In [ ]:
DefaultAssay(combined_obj) <- "ATAC_unified"
combined_obj <- RunHarmony(
  object = combined_obj,
  group.by.vars = c("patient"),
  reduction = "lsi.common_filt",
  assay.use = "ATAC_unified",
  project.dim = F,
  reduction.save = "harmony_lsi.common"
)

In [ ]:
combined_obj <- RunUMAP(
  combined_obj,
  dims = 1:19,
  reduction = "harmony_lsi.common",
  reduction.name = "umap.atac.harmony",
  reduction.key = "ATAChar_",
  assay = "ATAC_unified")

In [ ]:
combined_obj[["pca"]] <- NULL
combined_obj[["pca_filt"]] <- NULL
combined_obj[["lsi.common"]] <- NULL
combined_obj[["lsi.common_filt"]] <- NULL
combined_obj[["umap.atac.harmony"]] <- NULL
combined_obj[["umap.rna.harmony"]] <- NULL

### Joint 

In [ ]:
combined_obj <- FindMultiModalNeighbors(
 combined_obj,
 reduction.list = list("harmony_pca", "harmony_lsi.common"),
 dims.list = list(rna_comp, 1:19)
)
combined_obj <- RunUMAP(
   combined_obj,
   nn.name = "weighted.nn",
   reduction.name = "wnn.umap",
   reduction.key = "wnnUMAP_"
)

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 6)
p1 <- DimPlot(
  combined_obj,
   reduction = "wnn.umap",
   group.by = "patient",
   pt.size = 0.1,
   cols = colors_patients,
   raster=FALSE
) + ggtitle('Joint UMAP WNN\nby patient (Harmony)') + 
    theme(plot.title = element_text(hjust = 0.5, size = 12, face = "bold")) 

p2 <- DimPlot(
  combined_obj,
   reduction = "wnn.umap",
   group.by = "subproj",
   pt.size = 0.1,
   raster=FALSE
) + ggtitle('Joint UMAP WNN\nby subproject (Harmony)') + 
    theme(plot.title = element_text(hjust = 0.5, size = 12, face = "bold"))

p3 <- DimPlot(
  combined_obj,
   reduction = "wnn.umap",
   group.by = "cell_groups",
   pt.size = 0.1,
   cols=colors_cellg,
   raster=FALSE
) + ggtitle('Joint UMAP WNN\nby cell groups (Harmony)') + 
    theme(plot.title = element_text(hjust = 0.5, size = 12, face = "bold"))

#patchwork::wrap_plots(list(p1,p2,p3), ncol=3)

In [ ]:
combined_obj <- SetIdent(combined_obj, value=combined_obj@meta.data$patient)

In [ ]:
saveRDS(combined_obj, here(glue("{int}/{robj_dir}/4.0.3-integrated_harmony_object_{op}.rds")))

## Session

In [ ]:
sessionInfo()